# SepsisGuard — GRPO Training (SageMaker)

Train 4 multi-agent roles (Nurse, Lab, Pharmacist, Physician) using TRL GRPO with Unsloth 4-bit quantization.

**Kernel:** `conda_pytorch_p310`  
**Run from:** SageMaker Notebook Instance (ml.g4dn.xlarge or ml.g5.xlarge)  
**See:** `SAGEMAKER_GUIDE.md` for setup steps and cost estimates.

In [ ]:
# conda_pytorch_p310 already has PyTorch + CUDA — only extra packages needed.
!pip install -q -U "unsloth[colab-new]" "trl>=0.12" datasets requests httpx

In [ ]:
import os



# ── S3 destination for model artifacts ───────────────────────────────────────
# Leave S3_BUCKET as None to use the SageMaker default bucket automatically.
S3_BUCKET = None          # e.g. "my-sepsisguard-bucket"
S3_PREFIX = "sepsisguard" # folder inside the bucket 

# Resolve default S3 bucket if not overridden
if S3_BUCKET is None:
    import boto3
    account = boto3.client("sts").get_caller_identity()["Account"]
    region  = boto3.session.Session().region_name
    S3_BUCKET = f"sagemaker-{region}-{account}"

print(f"Model artifacts will be saved to: s3://{S3_BUCKET}/{S3_PREFIX}/")

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=4096, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
FastLanguageModel.for_inference(model)
print(f"Model loaded: {MODEL_NAME}")

In [ ]:
import os, sys

REPO_DIR = "/home/ec2-user/SageMaker/Sepsis-Guard"

if not os.path.exists(REPO_DIR):
    !git clone https://huggingface.co/spaces/Jishnu-Vijayan-03/Sepsis-Guard {REPO_DIR}

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!pip install -q -e "{REPO_DIR}"
print("Repo ready:", os.listdir(REPO_DIR))

In [ ]:
import os, signal, subprocess, time, requests

UVICORN_CMD = "uvicorn server.app:app --host 0.0.0.0 --port 7860"
HEALTH_URL  = "http://127.0.0.1:7860/health"

def _find_pids():
    proc = subprocess.run(["bash", "-lc", f"pgrep -f '{UVICORN_CMD}'"],
                          capture_output=True, text=True)
    if proc.returncode != 0 or not proc.stdout.strip(): return []
    return [int(x) for x in proc.stdout.strip().splitlines() if x.strip().isdigit()]

def _stop():
    for pid in _find_pids():
        try: os.kill(pid, signal.SIGTERM)
        except OSError: pass
    time.sleep(1.5)
    for pid in _find_pids():
        try: os.kill(pid, signal.SIGKILL)
        except OSError: pass

_stop()
subprocess.run(["bash", "-lc",
    f"nohup {UVICORN_CMD} > /tmp/uvicorn.log 2>&1 &"], check=False)

healthy = False
for _ in range(30):
    try:
        if requests.get(HEALTH_URL, timeout=3).ok:
            healthy = True; break
    except Exception: pass
    time.sleep(1)

print("Server healthy" if healthy else "Server failed — check /tmp/uvicorn.log")

In [ ]:
!curl -s "http://127.0.0.1:7860/health"

In [ ]:
import os, requests, time

ENV_URL = "http://127.0.0.1:7860"

class EnvClient:
    def __init__(self, base_url):
        self.base_url = base_url.rstrip("/")

    def reset(self, task_name, seed, session_id=None):
        h = {"X-Session-Id": session_id} if session_id else {}
        r = requests.post(f"{self.base_url}/reset",
                          json={"task_name": task_name, "seed": seed},
                          headers=h, timeout=30)
        r.raise_for_status(); return r.json()

    def step(self, actions, session_id=None):
        h = {"X-Session-Id": session_id} if session_id else {}
        r = requests.post(f"{self.base_url}/step",
                          json={"actions": actions}, headers=h, timeout=30)
        r.raise_for_status(); return r.json()

    def create_session(self):
        r = requests.post(f"{self.base_url}/session", timeout=10)
        r.raise_for_status(); return r.json()["session_id"]

    def delete_session(self, session_id):
        try: requests.delete(f"{self.base_url}/session/{session_id}", timeout=5)
        except Exception: pass

env  = EnvClient(ENV_URL)
info = env.reset(task_name="task1_textbook", seed=42)
print(f"Connected to {ENV_URL}")
print(f"Tick: {info['info']['tick']}, Roles: {list(info['observations'].keys())}")

In [ ]:
import os, sys, json
from tqdm.auto import tqdm
sys.path.insert(0, "/home/ec2-user/SageMaker/Sepsis-Guard")

from training.prompts import build_role_prompt
from agents.nurse import HeuristicNurse
from agents.lab import HeuristicLab
from agents.pharmacist import HeuristicPharmacist
from agents.physician import HeuristicPhysician

ROLES = ("nurse", "lab", "pharmacist", "physician")
N_EPISODES    = 16
SEEDS         = list(range(42, 42 + N_EPISODES))
TASK          = "task1_textbook"
MAX_TICKS     = 48

heuristic_agents = {
    "nurse":      HeuristicNurse(),
    "lab":        HeuristicLab(),
    "pharmacist": HeuristicPharmacist(),
    "physician":  HeuristicPhysician(),
}

# GRPO only needs prompts — it generates its own completions from the policy.
# Heuristic agents step through episodes in ~30 seconds.
print(f"Collecting prompts from {N_EPISODES} heuristic episodes...")
rollouts = []

for seed in tqdm(SEEDS, desc="Episodes"):
    sid    = env.create_session()
    bundle = env.reset(task_name=TASK, seed=seed, session_id=sid)
    done   = False
    tick   = 0
    while not done and tick < MAX_TICKS:
        obs     = bundle["observations"]
        actions = {}
        for role in ROLES:
            rollouts.append({"prompt": build_role_prompt(obs[role], role),
                             "role": role, "seed": seed, "tick": tick + 1})
            actions[role] = heuristic_agents[role].decide(obs[role])
        bundle = env.step(actions, session_id=sid)
        done   = bool(bundle.get("done", False))
        tick  += 1
    env.delete_session(sid)

print(f"\nCollected {len(rollouts)} prompts from {N_EPISODES} episodes")
print(f"Roles: {set(r['role'] for r in rollouts)}")
print(f"Avg ticks/episode: {len(rollouts) / N_EPISODES / len(ROLES):.0f}")

In [ ]:
import json, requests, re, torch
from datasets import Dataset
from training.prompts import build_role_prompt

# --- Smart deduplication ---
def get_clinical_state(prompt_str):
    try:
        obs_start = prompt_str.find("Observation:\n") + 13
        obs_end   = prompt_str.rfind("\nAction (JSON):")
        if obs_start < 13 or obs_end == -1: return prompt_str
        obs_dict  = json.loads(prompt_str[obs_start:obs_end])
        for key in ["tick", "reward", "cumulative_reward", "last_action_result",
                    "normalized_score", "done", "metadata"]:
            obs_dict.pop(key, None)
        return prompt_str[:obs_start] + json.dumps(obs_dict, sort_keys=True) + prompt_str[obs_end:]
    except Exception:
        return prompt_str

unique_prompts = {}
for r in rollouts:
    core = get_clinical_state(r["prompt"])
    if core not in unique_prompts:
        unique_prompts[core] = r["prompt"]

train_dataset = Dataset.from_list([{"prompt": p} for p in unique_prompts.values()])
print(f"Original rollouts:       {len(rollouts)}")
print(f"Unique clinical scenarios: {len(unique_prompts)}")

# --- Pre-training evaluation (before GRPO changes the model) ---
def run_episode(env_client, task_name, seed, agent_fn, session_id=None):
    bundle = env_client.reset(task_name=task_name, seed=seed, session_id=session_id)
    done   = False
    while not done:
        obs     = bundle["observations"]
        actions = {role: agent_fn(role, obs[role])
                   for role in ("nurse", "lab", "pharmacist", "physician")}
        bundle  = env_client.step(actions, session_id=session_id)
        done    = bundle["done"]
    grader = requests.get(f"{env_client.base_url}/grader",
                          headers={"X-Session-Id": session_id} if session_id else {},
                          timeout=30).json()
    return grader.get("score", 0.0)

def heuristic_agent_fn(role, obs):
    return heuristic_agents[role].decide(obs)

def make_llm_agent_fn(model, tokenizer, target_role):
    def agent_fn(role, obs):
        if role != target_role:
            return heuristic_agents[role].decide(obs)
        prompt = build_role_prompt(obs, role)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                skip_special_tokens=True)
        try:
            parsed = json.loads(text.strip())
            if isinstance(parsed, dict) and "operation" in parsed: return parsed
        except Exception: pass
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
                if isinstance(parsed, dict) and "operation" in parsed: return parsed
            except Exception: pass
        return heuristic_agents[role].decide(obs)
    return agent_fn

N_EVAL     = 5
EVAL_SEEDS = list(range(100, 100 + N_EVAL))

FastLanguageModel.for_inference(model)

print("\n" + "=" * 60)
print("PRE-TRAINING EVALUATION")
print("=" * 60)

pre_baseline = []
for seed in EVAL_SEEDS:
    sid = env.create_session()
    pre_baseline.append(run_episode(env, "task1_textbook", seed, heuristic_agent_fn, sid))
    env.delete_session(sid)
print(f"Heuristic baseline: mean={sum(pre_baseline)/len(pre_baseline):.4f}  "
      f"scores={[round(s,3) for s in pre_baseline]}")

pre_trained_scores = {}
for target_role in ("nurse", "lab", "pharmacist", "physician"):
    llm_fn = make_llm_agent_fn(model, tokenizer, target_role)
    scores = []
    for seed in EVAL_SEEDS:
        sid = env.create_session()
        scores.append(run_episode(env, "task1_textbook", seed, llm_fn, sid))
        env.delete_session(sid)
    pre_trained_scores[target_role] = scores
    print(f"Pre-train [{target_role}]: mean={sum(scores)/len(scores):.4f}  "
          f"scores={[round(s,3) for s in scores]}")
print("=" * 60)

In [ ]:
import torch, json, re, requests
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback
from training.reward_shaping import make_online_sepsis_reward_fn, format_reward_fn

FastLanguageModel.for_training(model)

# --- Hardware ---
has_cuda = torch.cuda.is_available()
if has_cuda:
    major, _ = torch.cuda.get_device_capability()
    use_bf16  = major >= 8   # A10G (ml.g5) and above
    use_fp16  = not use_bf16
    print(f"GPU: {torch.cuda.get_device_name(0)} | bf16={use_bf16} fp16={use_fp16}")
else:
    use_bf16 = use_fp16 = False

# --- GRPO config ---
# Checkpoints go to SageMaker persistent storage so they survive kernel restarts.
CKPT_DIR = "/home/ec2-user/SageMaker/sepsis-grpo-checkpoints"

cfg = GRPOConfig(
    output_dir                  = CKPT_DIR,
    num_generations             = 6,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    max_steps                   = 300,
    learning_rate               = 5e-6,
    warmup_steps                = 30,
    logging_steps               = 1,
    save_steps                  = 50,
    max_prompt_length           = 3000,
    max_completion_length       = 256,
    bf16                        = use_bf16,
    fp16                        = use_fp16,
    report_to                   = "none",
    gradient_checkpointing      = True,
)

# --- Reward functions ---
reward_fn_obj = make_online_sepsis_reward_fn(
    env_url      = ENV_URL,
    task_name    = TASK,
    seed         = 42,
    warmup_ticks = 4,
    inject_ticks = 4,
    max_workers  = 4,
)

def reward_fn_env(*args, **kwargs): return reward_fn_obj(*args, **kwargs)
reward_fn_env.__name__ = "reward_fn_env"

reward_log = {"steps": [], "env_reward": [], "format_reward": [], "combined_reward": []}

class RewardLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        step  = state.global_step
        env_r = (logs.get("rewards/reward_fn_env/mean") or logs.get("reward_fn_env", 0.0))
        fmt_r = (logs.get("rewards/format_reward_fn/mean") or logs.get("format_reward_fn", 0.0))
        combined = logs.get("reward", env_r + fmt_r)
        if env_r != 0.0 or "reward" in logs:
            reward_log["steps"].append(step)
            reward_log["env_reward"].append(float(env_r))
            reward_log["format_reward"].append(float(fmt_r))
            reward_log["combined_reward"].append(float(combined))
        if step % 50 == 0 and step > 0:
            try:
                r = requests.get(f"{ENV_URL}/health", timeout=5)
                if not r.ok: print(f"[WARN step {step}] Server unhealthy: {r.status_code}")
            except Exception as e:
                print(f"[WARN step {step}] Server unreachable: {e}")
            # Sample generation
            sample_prompt = train_dataset[0]["prompt"]
            inputs = tokenizer(sample_prompt, return_tensors="pt").to(model.device)
            model.eval()
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
            model.train()
            generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                         skip_special_tokens=True)
            match = re.search(r'\{.*?\}', generated, re.DOTALL)
            clean = match.group(0) if match else "No JSON found"
            warn  = ""
            if '"..."' in clean or '"rationale": ""' in clean: warn += " [WARNING: mode collapse]"
            if '"noop"' in clean or '"do_nothing"' in clean:      warn += " [WARNING: noop action]"
            print(f"\n=== Step {step}{warn} ===")
            print(f"Extracted: {clean[:300]}")
            if reward_log["steps"]:
                print(f"Latest: env={reward_log['env_reward'][-1]:.3f} "
                      f"fmt={reward_log['format_reward'][-1]:.3f}")
            print("=" * 40)

trainer = GRPOTrainer(
    model        = model,
    reward_funcs = [reward_fn_env, format_reward_fn],
    args         = cfg,
    train_dataset= train_dataset,
    callbacks    = [RewardLogger()],
)

print(f"Training: {cfg.max_steps} steps, lr={cfg.learning_rate}, "
      f"batch={cfg.per_device_train_batch_size}x{cfg.gradient_accumulation_steps}, "
      f"gen={cfg.num_generations}")
print(f"Checkpoints → {CKPT_DIR}")
print("Starting GRPO training...")
trainer.train()
print("Training complete.")

In [ ]:
import os, shutil, boto3

# --- Save locally first (to persistent SageMaker storage) ---
LOCAL_LORA   = "/home/ec2-user/SageMaker/sepsis-grpo-lora"
LOCAL_MERGED = "/home/ec2-user/SageMaker/sepsis-grpo-merged"

print(f"Saving LoRA adapters to {LOCAL_LORA} ...")
model.save_pretrained(LOCAL_LORA)
tokenizer.save_pretrained(LOCAL_LORA)

print(f"Saving merged 16-bit model to {LOCAL_MERGED} ...")
print("(This takes 5–10 minutes — do not interrupt)")
model.save_pretrained_merged(LOCAL_MERGED, tokenizer, save_method="merged_16bit")
print("Local save complete.")

# --- Upload to S3 ---
s3 = boto3.client("s3")

def upload_dir(local_dir, s3_prefix):
    for root, dirs, files in os.walk(local_dir):
        for fname in files:
            local_path = os.path.join(root, fname)
            rel_path   = os.path.relpath(local_path, local_dir)
            s3_key     = f"{s3_prefix}/{rel_path}"
            s3.upload_file(local_path, S3_BUCKET, s3_key)
            print(f"  uploaded: s3://{S3_BUCKET}/{s3_key}")

print(f"\nUploading LoRA to s3://{S3_BUCKET}/{S3_PREFIX}/lora/ ...")
upload_dir(LOCAL_LORA, f"{S3_PREFIX}/lora")

print(f"\nUploading merged model to s3://{S3_BUCKET}/{S3_PREFIX}/merged/ ...")
upload_dir(LOCAL_MERGED, f"{S3_PREFIX}/merged")

print(f"\nDone. Model artifacts at:")
print(f"  s3://{S3_BUCKET}/{S3_PREFIX}/lora/")
print(f"  s3://{S3_BUCKET}/{S3_PREFIX}/merged/")
print(f"\nTIP: Stop the notebook instance now if you are done training.")
print(f"     SageMaker -> Notebook instances -> Actions -> Stop")

In [ ]:
import requests, json, re, torch
from tqdm.auto import tqdm

FastLanguageModel.for_inference(model)

print("=" * 60)
print("POST-TRAINING EVALUATION")
print("=" * 60)

post_trained_scores = {}
for target_role in ("nurse", "lab", "pharmacist", "physician"):
    llm_fn = make_llm_agent_fn(model, tokenizer, target_role)
    scores = []
    for seed in EVAL_SEEDS:
        sid = env.create_session()
        scores.append(run_episode(env, "task1_textbook", seed, llm_fn, sid))
        env.delete_session(sid)
    post_trained_scores[target_role] = scores
    pre_mean  = sum(pre_trained_scores[target_role]) / len(pre_trained_scores[target_role])
    post_mean = sum(scores) / len(scores)
    print(f"Post-train [{target_role}]: mean={post_mean:.4f}  "
          f"delta_vs_pre={post_mean - pre_mean:+.4f}  "
          f"scores={[round(s,3) for s in scores]}")

# All-LLM evaluation
llm_fns = {role: make_llm_agent_fn(model, tokenizer, role)
           for role in ("nurse", "lab", "pharmacist", "physician")}
def all_llm_fn(role, obs): return llm_fns[role](role, obs)

all_llm_scores = []
for seed in EVAL_SEEDS:
    sid = env.create_session()
    all_llm_scores.append(run_episode(env, "task1_textbook", seed, all_llm_fn, sid))
    env.delete_session(sid)
print(f"\nAll-LLM: mean={sum(all_llm_scores)/len(all_llm_scores):.4f}  "
      f"scores={[round(s,3) for s in all_llm_scores]}")

# Summary table
heuristic_mean = sum(pre_baseline) / len(pre_baseline)
print(f"\n{'Role':<15} {'Heuristic':>10} {'Pre-Train':>10} {'Post-Train':>10} {'Delta':>10}")
print("-" * 60)
for role in ("nurse", "lab", "pharmacist", "physician"):
    pre  = sum(pre_trained_scores[role]) / len(pre_trained_scores[role])
    post = sum(post_trained_scores[role]) / len(post_trained_scores[role])
    print(f"{role:<15} {heuristic_mean:>10.4f} {pre:>10.4f} {post:>10.4f} {post-pre:>+10.4f}")
print("=" * 60)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Panel 1: Reward curves
if reward_log["steps"]:
    axes[0].plot(reward_log["steps"], reward_log["env_reward"],    label="Env Reward",    linewidth=1.5, alpha=0.8)
    axes[0].plot(reward_log["steps"], reward_log["format_reward"], label="Format Reward", linewidth=1.5, alpha=0.8)
    if reward_log["combined_reward"]:
        axes[0].plot(reward_log["steps"], reward_log["combined_reward"],
                     label="Combined", linewidth=2, color="black", alpha=0.5)
    axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.3)
    axes[0].set_xlabel("Training Step")
    axes[0].set_ylabel("Mean Reward")
    axes[0].set_title("GRPO Training Reward Curves")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, "No reward logs captured", ha="center", va="center",
                 transform=axes[0].transAxes)
    axes[0].set_title("GRPO Training Reward Curves")

# Panel 2: Pre vs post per role
roles = ["nurse", "lab", "pharmacist", "physician"]
pre_means  = [sum(pre_trained_scores[r])  / len(pre_trained_scores[r])  for r in roles]
post_means = [sum(post_trained_scores[r]) / len(post_trained_scores[r]) for r in roles]
x = np.arange(len(roles)); width = 0.35
bars_pre  = axes[1].bar(x - width/2, pre_means,  width, label="Pre-Training",  color="#FF9800", alpha=0.8)
bars_post = axes[1].bar(x + width/2, post_means, width, label="Post-Training", color="#4CAF50", alpha=0.8)
axes[1].axhline(y=heuristic_mean, color="#888", linestyle="--", alpha=0.7,
                label=f"Heuristic ({heuristic_mean:.3f})")
axes[1].set_xticks(x); axes[1].set_xticklabels([r.capitalize() for r in roles])
axes[1].set_ylabel("Episode Score"); axes[1].set_title("Pre vs Post Training")
axes[1].set_ylim(0, 1.0); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars_pre,  pre_means):  axes[1].text(bar.get_x()+bar.get_width()/2, val+0.02, f"{val:.2f}", ha="center", fontsize=7)
for bar, val in zip(bars_post, post_means): axes[1].text(bar.get_x()+bar.get_width()/2, val+0.02, f"{val:.2f}", ha="center", fontsize=7)

# Panel 3: Delta
deltas = [post - pre for post, pre in zip(post_means, pre_means)]
colors = ["#4CAF50" if d >= 0 else "#F44336" for d in deltas]
bars_d = axes[2].bar(roles, deltas, color=colors, alpha=0.8)
axes[2].axhline(y=0, color="gray", alpha=0.5)
axes[2].set_ylabel("Score Change (Post - Pre)"); axes[2].set_title("Improvement by Role")
axes[2].grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars_d, deltas):
    axes[2].text(bar.get_x()+bar.get_width()/2,
                 val+0.01 if val >= 0 else val-0.03,
                 f"{val:+.3f}", ha="center",
                 va="bottom" if val >= 0 else "top", fontsize=9)

plt.tight_layout()
plot_path = "/home/ec2-user/SageMaker/training_results.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to {plot_path}")